# 미국주식 confound 해소 실험 — 임베딩 기반 단일 정책 재검증

`us_equity_hrl_generalization_colab.ipynb`은 시장(미국 대형주)과 상태표현(원시 가격윈도우, 임베딩
없음)을 동시에 바꿔서, "단일 정책 실패가 재현 안 된 것"이 market 때문인지 architecture 때문인지
가릴 수 없었다(`paper/DRAFT.md`/`DRAFT_KR.md` 1.1절·7장·8장 한계#1). 이 노트북은 **market을
고정하고 architecture만 크립토/KOSPI와 맞춰서** 이 confound를 직접 해소한다.

**설계**: 같은 44종목·같은 2024–2025 테스트 구간에, 원시 가격윈도우 대신 **Chronos T5 zero-shot
임베딩**(사전학습됨, 추가 학습 불필요 — `crypto_hybrid_model_colab.ipynb` §2와 동일 추출 코드
재사용)을 상태로 쓰고, `crypto_hrl_earnhft_colab.ipynb`의 `CryptoPortfolioEnv`(임베딩 + 현재비중
→ dense softmax, 클러스터링도 CNN도 없음 — 크립토 원본 단일PPO·KOSPI 실패가 재현됐던 것과 동일한
"밀집 임베딩 아키텍처")를 그대로 재사용한다. MTGNN 학습(수 시간)은 생략 — 원래 백본 비교에서
Chronos zero-shot **단독**이 방향성 정확도 1위였다는 기록(`progress_report_final.md`)이 있어,
MTGNN 없이도 confound를 가리기에 충분한 "임베딩 기반" 상태를 얻을 수 있다는 근거가 있다.

**판정**: 이 설정에서도 단일 정책 PPO가 등가중을 이긴다(=`us_equity_hrl_generalization_colab.ipynb`의
결과와 같음) → market이 더 유력한 설명. 여기서는 실패한다(=크립토/KOSPI와 같음) → architecture가
더 유력한 설명, 이 프로젝트의 confound 가설이 확정됨.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q chronos-forecasting stable-baselines3[extra] gymnasium pyarrow scikit-learn


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. 설정

In [2]:
import warnings
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['axes.unicode_minus'] = False

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

DRIVE_ROOT = Path('/content/drive/MyDrive/졸업프로젝트')
DATA_DIR   = DRIVE_ROOT / 'data' / 'us_equity'
MODEL_DIR  = DRIVE_ROOT / 'models' / 'us_equity_embedding'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

CACHE_PATH = DATA_DIR / 'ohlc_raw.parquet'
EMB_CACHE  = DATA_DIR / 'us_equity_chronos_embs_pca64.npz'
assert CACHE_PATH.exists(), (
    'ohlc_raw.parquet 없음 — us_equity_hrl_generalization_colab.ipynb §2를 먼저 실행해 '
    '가격 데이터를 캐시해야 함(이 노트북은 그 캐시를 재사용).'
)

SEQ_LEN = 60      # 60거래일 윈도우 — crypto_hybrid_model_colab.ipynb와 동일 관례(길이 단위만 봉→일)
CHRONOS_DIM = 512  # chronos-t5-small hidden dim
EMB_DIM = 64       # PCA 축소 후 차원 — 크립토/KOSPI의 RL 임베딩과 동일 차원(직접 비교 가능하게)
TC = 0.001

TRAIN_START = '2015-01-01'
TRAIN_END   = '2022-12-31'
VALID_START = '2023-01-01'
VALID_END   = '2023-12-31'
TEST_START  = '2024-01-01'
TEST_END    = '2025-12-31'

SINGLE_PPO_TIMESTEPS = 100_000   # 원시 가격윈도우 버전과 동일 예산(직접 비교 가능하게)

print('설정 완료')


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Device: cpu
설정 완료


## 2. 가격 데이터 로드 (캐시 재사용)

In [3]:
raw = pd.read_parquet(CACHE_PATH)
all_symbols = sorted({c[0] for c in raw.columns})
print(f'캐시에서 {len(all_symbols)}종목 로드')

all_bdays = pd.bdate_range(TRAIN_START, TEST_END)
min_len = int(len(all_bdays) * 0.95)

symbols, per_symbol = [], {}
for sym in all_symbols:
    try:
        df = raw[sym][['Close']].dropna()
    except KeyError:
        continue
    if len(df) < min_len:
        continue
    per_symbol[sym] = df
    symbols.append(sym)

N_ASSETS = len(symbols)
print(f'유효 종목: {N_ASSETS}')

common_index = None
for sym in symbols:
    idx = per_symbol[sym].index
    common_index = idx if common_index is None else common_index.union(idx)
common_index = common_index.sort_values()

close_mat = np.full((N_ASSETS, len(common_index)), np.nan, dtype=np.float32)
for i, sym in enumerate(symbols):
    close_mat[i] = per_symbol[sym].reindex(common_index).ffill().bfill()['Close'].to_numpy(dtype=np.float32)
assert not np.isnan(close_mat).any()

pos_of = {d: i for i, d in enumerate(common_index)}
print(f'가격 행렬: {close_mat.shape}, {common_index[0].date()} ~ {common_index[-1].date()}')

ret_index = {}
for i, sym in enumerate(symbols):
    c = close_mat[i]
    fwd = c[1:] / c[:-1] - 1
    for pos in range(len(fwd)):
        ret_index[(sym, pos)] = float(fwd[pos])


캐시에서 44종목 로드
유효 종목: 44
가격 행렬: (44, 2765), 2015-01-02 ~ 2025-12-30


## 3. Chronos T5 zero-shot 임베딩 추출 (학습 없음, 추론만 — 최초 1회 후 캐시)

In [4]:
%%time
from chronos import ChronosPipeline

if EMB_CACHE.exists():
    cached = np.load(EMB_CACHE, allow_pickle=True)
    embs_pca64 = cached['embs']
    positions_with_emb = cached['positions']
    print(f'캐시 로드: {embs_pca64.shape}')
else:
    gc.collect()
    torch.cuda.empty_cache()
    pipeline = ChronosPipeline.from_pretrained(
        'amazon/chronos-t5-small', device_map=DEVICE, torch_dtype=torch.float32,
    )
    print('Chronos 로드 완료')

    @torch.no_grad()
    def extract_chronos_embs(close_prices_np, batch_size=256):
        tokenizer = pipeline.tokenizer
        encoder   = pipeline.model.model.encoder
        all_embs  = []
        for i in range(0, len(close_prices_np), batch_size):
            ctx = torch.tensor(close_prices_np[i:i + batch_size], dtype=torch.float32)
            ids, mask, _ = tokenizer.context_input_transform(ctx)
            out = encoder(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE))
            h = out.last_hidden_state
            m = mask.to(DEVICE).unsqueeze(-1).float()
            emb = (h * m).sum(1) / m.sum(1).clamp(min=1)
            all_embs.append(emb.cpu().float().numpy())
        return np.concatenate(all_embs, axis=0)

    # 종목×포지션 전부에 대해 60일 트레일링 윈도우 임베딩 추출 (워밍업 SEQ_LEN 이후만)
    all_positions = list(range(SEQ_LEN, len(common_index)))
    flat_windows, flat_keys = [], []
    for i, sym in enumerate(symbols):
        for pos in all_positions:
            flat_windows.append(close_mat[i, pos - SEQ_LEN:pos])
            flat_keys.append((sym, pos))
    flat_windows = np.stack(flat_windows).astype(np.float32)
    print(f'임베딩 추출 대상: {len(flat_windows):,}개 윈도우 ({len(symbols)}종목 × {len(all_positions)}일)')

    raw_embs = extract_chronos_embs(flat_windows)
    print(f'원시 Chronos 임베딩: {raw_embs.shape}')

    # PCA로 512 -> 64차원 축소 — 크립토/KOSPI RL 임베딩과 동일 차원으로 맞춰 직접 비교 가능하게
    pca = PCA(n_components=EMB_DIM, random_state=42)
    embs_pca64 = pca.fit_transform(raw_embs).astype(np.float32)
    print(f'PCA 축소 후: {embs_pca64.shape}, 설명 분산 비율 합계 = {pca.explained_variance_ratio_.sum():.3f}')

    positions_with_emb = np.array(flat_keys, dtype=object)
    np.savez(EMB_CACHE, embs=embs_pca64, positions=positions_with_emb)
    print('저장 완료:', EMB_CACHE)

emb_index = {(sym, int(pos)): i for i, (sym, pos) in enumerate(positions_with_emb)}
embs_all = embs_pca64
print(f'emb_index: {len(emb_index):,}개')


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Chronos 로드 완료
임베딩 추출 대상: 119,020개 윈도우 (44종목 × 2705일)
원시 Chronos 임베딩: (119020, 512)
PCA 축소 후: (119020, 64), 설명 분산 비율 합계 = 0.924
저장 완료: /content/drive/MyDrive/졸업프로젝트/data/us_equity/us_equity_chronos_embs_pca64.npz
emb_index: 119,020개
CPU times: user 1h 24min 29s, sys: 8min, total: 1h 32min 29s
Wall time: 1h 34min 19s


## 4. CryptoPortfolioEnv 재사용 — `crypto_hrl_earnhft_colab.ipynb` §3과 완전히 동일(임베딩+밀집 softmax)

In [5]:
# [2026-10-04 회계 수정] 가격 변동 후 실제 보유 비중(drift)·거래 가능 종목 마스크.
# 이전 버전은 수익 반영 뒤에도 직전 '목표' 비중을 보유 비중으로 저장해, 가격 변동으로 틀어진 비중을
# 되돌리는 재조정 거래(등가중 포함)의 비용이 0으로 계산됐다.
def _drift_weights(w, rets):
    """수익 반영 뒤 실제 보유 비중 w_i(1+r_i)/(1+wᵀr). 합<1이면 나머지는 현금(수익률 0)으로 남는다."""
    w = np.asarray(w, dtype=np.float64)
    g = 1.0 + float(np.dot(w, rets))
    if g <= 1e-12:
        return w.astype(np.float32)
    return (w * (1.0 + np.asarray(rets, dtype=np.float64)) / g).astype(np.float32)


def _mask_untradable(w, tradable, renorm=True):
    """수익률 데이터가 없는(상장 전·상폐 후) 종목 비중을 0으로. renorm=False면 빠진 몫은 현금으로 남는다."""
    w = np.where(tradable, np.asarray(w, dtype=np.float64), 0.0)
    s = w.sum()
    return (w / s if renorm and s > 1e-12 else w).astype(np.float32)


class CryptoPortfolioEnv(gym.Env):
    '''crypto_hrl_earnhft_colab.ipynb의 CryptoPortfolioEnv와 동일 로직(§0 고지 참고) —
    종목별 임베딩(N×64) + 현재 비중(N) -> N*64+N dim 상태, N차원 dense softmax 행동. 인덱스가
    30분 버킷이 아니라 거래일 포지션이라는 점만 다르다. 크립토 원본 단일PPO(-75.68%)·KOSPI
    단일PPO(+0.28%)가 실패를 보였던 것과 정확히 같은 아키텍처 계열이다.'''
    metadata = {'render_modes': []}

    def __init__(self, positions, symbols, emb_index, embs_all, ret_index, emb_dim=EMB_DIM, tc=TC):
        super().__init__()
        self.positions  = positions
        self.symbols    = symbols
        self.n_assets   = len(symbols)
        self.emb_index  = emb_index
        self.embs_all   = embs_all
        self.ret_index  = ret_index
        self.emb_dim    = emb_dim
        self.tc         = tc

        obs_dim = self.n_assets * emb_dim + self.n_assets
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(obs_dim,), dtype=np.float32)
        self.action_space      = spaces.Box(-3.0, 3.0, shape=(self.n_assets,), dtype=np.float32)

    def _asset_emb(self, pos):
        embs = []
        for sym in self.symbols:
            idx = self.emb_index.get((sym, pos))
            embs.append(self.embs_all[idx] if idx is not None else np.zeros(self.emb_dim, dtype=np.float32))
        return np.concatenate(embs)

    def _asset_ret(self, pos):
        return np.array([self.ret_index.get((sym, pos), 0.0) for sym in self.symbols], dtype=np.float32)

    def _get_obs(self):
        pos = self.positions[self.t]
        return np.concatenate([self._asset_emb(pos), self.weights]).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t                = 0
        self.weights           = np.ones(self.n_assets, dtype=np.float32) / self.n_assets
        self.portfolio_value   = 1.0
        self.value_history     = [1.0]
        self.turnover_history  = []
        return self._get_obs(), {}

    def step(self, action):
        action = np.exp(action - action.max())
        action = action / action.sum()

        turnover = float(np.abs(action - self.weights).sum())
        tc_cost  = self.tc * turnover

        pos = self.positions[self.t]
        asset_rets = self._asset_ret(pos)
        port_ret = float((action * asset_rets).sum())

        self.portfolio_value *= (1 + port_ret - tc_cost)
        self.weights = _drift_weights(action, asset_rets)  # 가격 변동 후 실제 보유 비중
        self.value_history.append(self.portfolio_value)
        self.turnover_history.append(turnover)
        self.t += 1

        done = (self.t >= len(self.positions) - 1)
        reward = float(port_ret - tc_cost)
        info = {'portfolio_value': self.portfolio_value, 'port_ret': port_ret, 'turnover': turnover}
        return self._get_obs(), reward, done, False, info


def date_range_positions(start, end):
    return [pos_of[d] for d in common_index if pd.Timestamp(start) <= d <= pd.Timestamp(end)]

train_positions = [p for p in date_range_positions(TRAIN_START, TRAIN_END) if p >= SEQ_LEN]
valid_positions  = [p for p in date_range_positions(VALID_START, VALID_END) if p >= SEQ_LEN]
test_positions   = [p for p in date_range_positions(TEST_START, TEST_END) if p >= SEQ_LEN]
print(f'train: {len(train_positions)}일, valid: {len(valid_positions)}일, test: {len(test_positions)}일')

env_kwargs = dict(symbols=symbols, emb_index=emb_index, embs_all=embs_all, ret_index=ret_index)
_env = CryptoPortfolioEnv(train_positions[:100], **env_kwargs)
check_env(_env, warn=True)
print('환경 유효성 검사 완료')


train: 1954일, valid: 250일, test: 501일
환경 유효성 검사 완료


## 5. 단일 정책 PPO 학습 — 원시 가격윈도우 버전과 동일 예산

In [6]:
%%time
def equal_weight_backtest(positions):
    env = CryptoPortfolioEnv(positions, **env_kwargs)
    obs, _ = env.reset()
    done = False
    action = np.zeros(env.n_assets, dtype=np.float32)  # softmax(0) = 균등
    while not done:
        obs, _, done, _, info = env.step(action)
    return np.array(env.value_history), list(env.turnover_history)


def perf_row(values, turnovers, tc=TC, steps_per_year=252):
    rets   = np.diff(values) / values[:-1]
    ann    = values[-1] ** (steps_per_year / len(rets)) - 1
    sharpe = rets.mean() / (rets.std() + 1e-8) * np.sqrt(steps_per_year)
    mdd    = ((values / np.maximum.accumulate(values)) - 1).min()
    t      = np.asarray(turnovers, dtype=np.float64)
    return {
        '총 수익률': f'{(values[-1]-1)*100:.2f}%', '연환산': f'{ann*100:.2f}%',
        'Sharpe': f'{sharpe:.3f}', 'MDD': f'{mdd*100:.2f}%',
        '평균 턴오버/스텝': f'{t.mean()*100:.2f}%',
    }


train_env = Monitor(CryptoPortfolioEnv(train_positions, **env_kwargs))
embedding_ppo = PPO(
    'MlpPolicy', train_env,
    learning_rate=3e-4, n_steps=1024, batch_size=64, n_epochs=10,
    gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.01, vf_coef=0.5,
    max_grad_norm=0.5, policy_kwargs=dict(net_arch=[256, 256]),
    verbose=0, device='cpu', seed=42,
)
embedding_ppo.learn(total_timesteps=SINGLE_PPO_TIMESTEPS, progress_bar=True)
embedding_ppo.save(str(MODEL_DIR / 'embedding_single_ppo_seed42.zip'))

def run_model(model, positions):
    env = CryptoPortfolioEnv(positions, **env_kwargs)
    obs, _ = env.reset()
    done = False
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        obs, _, done, _, info = env.step(a)
    return np.array(env.value_history), list(env.turnover_history)

ew_test_vals, ew_test_turns = equal_weight_backtest(test_positions)
ep_test_vals, ep_test_turns = run_model(embedding_ppo, test_positions)
ew_valid_vals, ew_valid_turns = equal_weight_backtest(valid_positions)
ep_valid_vals, ep_valid_turns = run_model(embedding_ppo, valid_positions)

print('=== [confound 해소] 임베딩 기반 단일 정책 PPO vs 등가중 ===\n')
print('--- valid(2023) ---')
print('등가중:      ', perf_row(ew_valid_vals, ew_valid_turns))
print('임베딩 단일PPO:', perf_row(ep_valid_vals, ep_valid_turns))
print('\n--- test(2024-2025) ---')
print('등가중:      ', perf_row(ew_test_vals, ew_test_turns))
print('임베딩 단일PPO:', perf_row(ep_test_vals, ep_test_turns))


Output()

=== [confound 해소] 임베딩 기반 단일 정책 PPO vs 등가중 ===

--- valid(2023) ---
등가중:       {'총 수익률': '19.38%', '연환산': '19.63%', 'Sharpe': '1.533', 'MDD': '-10.51%', '평균 턴오버/스텝': '0.00%'}
임베딩 단일PPO: {'총 수익률': '17.40%', '연환산': '17.62%', 'Sharpe': '1.382', 'MDD': '-10.49%', '평균 턴오버/스텝': '9.18%'}

--- test(2024-2025) ---
등가중:       {'총 수익률': '47.07%', '연환산': '21.46%', 'Sharpe': '1.477', 'MDD': '-16.66%', '평균 턴오버/스텝': '0.00%'}
임베딩 단일PPO: {'총 수익률': '37.78%', '연환산': '17.53%', 'Sharpe': '1.252', 'MDD': '-16.61%', '평균 턴오버/스텝': '9.41%'}
CPU times: user 11min 21s, sys: 4.26 s, total: 11min 25s
Wall time: 11min 51s


## 6. 판정 기준

1. **핵심**: test에서 임베딩 기반 단일 정책 PPO가 등가중 대비 크립토 원본(−75.68% vs +2.66%)·
   KOSPI(+0.28% vs +34.33%)처럼 큰 폭으로 실패하는가?
   - **실패한다** → market이 아니라 **architecture(임베딩+밀집 softmax)가 단일 정책 실패의
     더 유력한 원인**이라는 이 프로젝트의 confound 가설이 확정됨. `us_equity_hrl_generalization_colab.ipynb`의
     "실패 없음" 결과는 원시 가격윈도우 상태표현 덕분이었다는 뜻.
   - **여전히 등가중과 비슷하거나 이긴다** → architecture만으로는 설명이 안 되고, market/레짐
     (예: 2024-2025 미국 대형주의 낮은 자산 간 분산, 강한 상승 추세)이 더 유력한 원인일 가능성이
     커짐 — confound가 완전히 안 풀린 것이므로, 두 변수를 더 분리하는 후속 실험이 필요하다는
     결론.
2. 결과는 `paper/GAP_ANALYSIS.md`·`FINDINGS_LOG.md`·`DRAFT.md`/`DRAFT_KR.md` 1.1절·7절·8장
   한계#1의 confound 서술을 확정된 결론으로 갱신하는 데 쓴다.